size of each study

In [ ]:
import pandas as pd
import sys


def print_full_df2(x):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", None)
    pd.set_option("display.max_colwidth", None)
    display(x)
    pd.reset_option("display.max_rows")
    pd.reset_option("display.max_columns")
    pd.reset_option("display.width")
    pd.reset_option("display.float_format")
    pd.reset_option("display.max_colwidth")


sample_group = pd.read_csv(
    "../../data/sun_et_al_data/sample_group_species_preprocessed.csv"
)
print_full_df2(sample_group.head())

,Sample,Group,Group_1,Group_2,LevelB,LevelA,group,Project,Project_1,Project_2,LevelA2,LevelA3,LevelA4,LevelA5,Age,Gender,BMI,Location_1,Location,sequence_type,DNA_extract_type
0,K0060,Disease,Disease,Disease,IBS,digestive disease,IBS-D,HanL_2021.IBS,HanL_2021,Discover,digestive disease,digestive disease,IBS,digestive disease,NaN,NaN,NaN,Hongkong,Hongkong,Illumina HiSeq 2000,phenol/chloroform/isoamyl alcohol
1,SAMEA2737843,Control,Control,Control,RA,immune disease,Control,ZhangX_2015.RA,ZhangX_2015,Discover,immune disease,immune disease,RA,immune disease,NaN,NaN,NaN,Beijing,Beijing,Illumina HiSeq 2000,QinJ_2012
2,G150T0,Disease,Disease,Disease,Gout,immune disease,Gout,ChuY_2021.Gout,ChuY_2021,Discover,immune disease,immune disease,Gout,immune disease,26.0,Male,26.423570,Guangzhou,Guangzhou (Guangdong),Illumina HiSeq 4000,E.Z.N.A Stool DNA Kit
3,ERR2855942,Control,Control,Control,SCZ,mental disease,healthy,ZhuF_2020.SCZ,ZhuF_2020,Discover,Others,mental disease,SCZ,Others,24.0,Male,20.715694,Xi'an,Xi¡¯an (Shannxi),Illumina / BGISEQ-500,QIAamp DNA Stool Mini Kit
4,SRR6504865,Disease,Disease,Disease,CD,digestive disease,CD,WengY_2019.CD,WengY_2019,Discover,digestive disease,digestive disease,IBD,digestive disease,NaN,NaN,NaN,Guangzhou,Guangzhou (Guangdong),Illumina HiSeq X Ten,"NEBNext Ultra DNA Library Prep Kit for Illumina (New England Biolabs, Ipswich, MA, USA)"


In [ ]:
group_size_per_label_per_study = {}
grouped_by_study = sample_group.groupby("Project_1")
for name, group in grouped_by_study:
    group_size_per_label_per_study[name] = {}
    grouped_by_disease = group.groupby("group")
    for dis, g in grouped_by_disease:
        group_size_per_label_per_study[name][dis] = len(g)

display(group_size_per_label_per_study)

{'ChenB_2020': {'Control': 116, 'SLE': 115},
 'ChuY_2021': {'Control': 85, 'Gout': 95},
 'HanL_2021': {'Control': 84, 'IBS-D': 277},
 'HeQ_2017': {'CD': 46, 'Control': 52},
 'HuY_2019': {'Health': 31, 'Patient': 30, 'Test-P': 16},
 'HuangR_2020': {'AS': 113, 'Control': 37},
 'JieZ_2017': {'AMI': 4,
  'Control': 160,
  'Stable angina': 193,
  'Unstable angina': 6},
 'LiJ_2017': {'Control': 41, 'HTN': 98, 'pHTN': 56},
 'LiR_2021': {'CA': 100, 'Control': 36},
 'LiuP_2021': {'Control': 49, 'MG': 74},
 'LiuR_2017': {'Control': 105, 'DB': 112},
 'LuW_2018': {'Control': 10, 'INR': 20, 'IR': 15, 'VU': 25},
 'MaoL_2021': {'CON': 39, 'PD': 39},
 'QiX_2019': {'Control': 43, 'PCOS': 49},
 'QianY_2020': {'MC': 40, 'MP': 40},
 'QinJ_2012': {'Control': 183, 'T2D': 186},
 'QinN_2014': {'Control': 114, 'liver': 117},
 'WanY_2021': {'ASD': 64, 'Control': 64},
 'WangM_2019': {'ASD': 41, 'Control': 31},
 'WangQ_2021': {'low': 167, 'normal': 119, 'osteoporosis': 64},
 'WangX_2020': {'Control': 69, 'ESRD': 

Group sizes sorted by total group size:


,Study,Total Group Size
0,QinJ_2012,369
1,JieZ_2017,363
2,ZengQ_2021,363
3,HanL_2021,361
4,WangQ_2021,350
5,WangX_2020,275
6,ZhongH_2019,254
7,YeohYK_2021,249
8,ChenB_2020,231
9,QinN_2014,231


In [14]:
# plot count per study for the different diseases in that study as a bar plot (stacked)
# use plotly and write disease name on the corresponding bar (except for healthy)
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px

# Convert the nested dict to a long-format DataFrame
def plot_disease_counts(data):
    # Convert to a long format DataFrame
    records = []
    for study, conditions in data.items():
        for condition, count in conditions.items():
            records.append({
                'Study': study,
                'Condition': condition,
                'Count': count
            })

    df = pd.DataFrame(records)
    
    # Calculate total sample count for each study
    study_totals = df.groupby('Study')['Count'].sum().reset_index()

    # Sort studies by total count (largest first)
    studies = study_totals.sort_values('Count', ascending=False)['Study'].tolist()
    
    # Get unique conditions
    conditions = sorted(df['Condition'].unique())
    
    # Generate a color map for all conditions
    condition_colors = {}
    color_scale = px.colors.qualitative.Bold
    for i, condition in enumerate(conditions):
        condition_colors[condition] = color_scale[i % len(color_scale)]
    
    # Create the figure
    fig = go.Figure()
    
    # Track which conditions have been added to the legend
    conditions_in_legend = {}
    
    # For each study
    for study in studies:
        study_data = df[df['Study'] == study]
        
        # Add all condition bars with unique colors
        for _, row in study_data.iterrows():
            condition = row['Condition']
            # Only show in legend the first time
            show_in_legend = condition not in conditions_in_legend
            if show_in_legend:
                conditions_in_legend[condition] = True
                
            fig.add_trace(go.Bar(
                y=[study],
                x=[row['Count']],
                name=condition,
                orientation='h',
                marker=dict(color=condition_colors.get(condition, 'gray')),
                legendgroup=condition,
                showlegend=show_in_legend
            ))

    # Update layout
    fig.update_layout(
        barmode='stack',
        # title='Number of samples per disease per study',
        xaxis_title='Number of samples',
        yaxis_title='Study',
        legend_title='',
        height=700,  # Taller graph
        width=800,   # Narrower overall width
        margin=dict(l=10, r=20, t=50, b=50),  # More space for study names
        font=dict(size=12),  # Larger general text
    )
    
    # Make bars wider
    fig.update_traces(width=0.85)
    fig.update_layout(showlegend=False)
    
    return fig

plot_disease_counts(group_size_per_label_per_study).show()

In [5]:
print_full_df2(sample_group[sample_group["Project_1"] == "LuW_2018"])

,Sample,Group,Group_1,Group_2,LevelB,LevelA,group,Project,Project_1,Project_2,LevelA2,LevelA3,LevelA4,LevelA5,Age,Gender,BMI,Location_1,Location,sequence_type,DNA_extract_type
130,SRR5818512,Disease,Disease,Disease,HIV,infectious disease,IR,LuW_2018.HIV,LuW_2018,Discover,infectious disease,infectious disease,HIV,Others,47.0,Male,25.640243,Beijing,Beijing,Illumina HiSeq 2500,PSP® Spin Stool DNA Plus Kit
138,SRR5818471,Disease,Disease,Disease,HIV,infectious disease,VU,LuW_2018.HIV,LuW_2018,Discover,infectious disease,infectious disease,HIV,Others,28.0,Male,19.817677,Beijing,Beijing,Illumina HiSeq 2500,PSP® Spin Stool DNA Plus Kit
175,SRR5818478,Control,Control,Control,HIV,infectious disease,Control,LuW_2018.HIV,LuW_2018,Discover,infectious disease,infectious disease,HIV,Others,45.0,Male,23.719609,Beijing,Beijing,Illumina HiSeq 2500,PSP® Spin Stool DNA Plus Kit
212,SRR5818475,Control,Control,Control,HIV,infectious disease,Control,LuW_2018.HIV,LuW_2018,Discover,infectious disease,infectious disease,HIV,Others,26.0,Male,24.567474,Beijing,Beijing,Illumina HiSeq 2500,PSP® Spin Stool DNA Plus Kit
215,SRR5818484,Disease,Disease,Disease,HIV,infectious disease,VU,LuW_2018.HIV,LuW_2018,Discover,infectious disease,infectious disease,HIV,Others,34.0,Male,21.383942,Beijing,Beijing,Illumina HiSeq 2500,PSP® Spin Stool DNA Plus Kit
220,SRR5818498,Disease,Disease,Disease,HIV,infectious disease,INR,LuW_2018.HIV,LuW_2018,Discover,infectious disease,infectious disease,HIV,Others,28.0,Male,20.199470,Beijing,Beijing,Illumina HiSeq 2500,PSP® Spin Stool DNA Plus Kit
232,SRR5818502,Disease,Disease,Disease,HIV,infectious disease,INR,LuW_2018.HIV,LuW_2018,Discover,infectious disease,infectious disease,HIV,Others,54.0,Male,23.054562,Beijing,Beijing,Illumina HiSeq 2500,PSP® Spin Stool DNA Plus Kit
233,SRR5818511,Disease,Disease,Disease,HIV,infectious disease,VU,LuW_2018.HIV,LuW_2018,Discover,infectious disease,infectious disease,HIV,Others,61.0,Male,20.761246,Beijing,Beijing,Illumina HiSeq 2500,PSP® Spin Stool DNA Plus Kit
321,SRR5818494,Disease,Disease,Disease,HIV,infectious disease,VU,LuW_2018.HIV,LuW_2018,Discover,infectious disease,infectious disease,HIV,Others,33.0,Male,19.623234,Beijing,Beijing,Illumina HiSeq 2500,PSP® Spin Stool DNA Plus Kit
325,SRR5818507,Disease,Disease,Disease,HIV,infectious disease,INR,LuW_2018.HIV,LuW_2018,Discover,infectious disease,infectious disease,HIV,Others,43.0,Male,24.677021,Beijing,Beijing,Illumina HiSeq 2500,PSP® Spin Stool DNA Plus Kit
